In [1]:
import os
from kalpy.fstext.lexicon import LexiconCompiler
from kalpy.fstext.utils import kaldi_to_pynini, pynini_to_kaldi
from kalpy.gmm.align import GmmAligner
from kalpy.decoder.training_graphs import TrainingGraphCompiler
from kalpy.feat.data import FeatureArchive
import pywrapfst
import pynini
from _kalpy.fstext import TableCompose, VectorFst, MakeLinearAcceptor


In [2]:
dict_name = "1_english_uk_mfa"

lexicon_fst_path = rf"D:\temp\MFA\english\dictionary\{dict_name}\L.fst"
align_lexicon_fst_path = rf"D:\temp\MFA\english\dictionary\{dict_name}\align_lexicon.fst"
words_symbol_path = rf"D:\temp\MFA\english\dictionary\{dict_name}\words.txt"
phone_symbol_path = r"D:\temp\MFA\english\dictionary\phones\phones.txt"

lexicon_compiler = LexiconCompiler(
    silence_probability=0.5,
    initial_silence_probability=0.5,
    final_silence_correction=None,
    final_non_silence_correction=None,
    silence_word = "<eps>",
    oov_word = "<unk>",
    silence_phone = "sil",
    oov_phone = "spn",
    position_dependent_phones = False,
)
lexicon_compiler.load_l_from_file(lexicon_fst_path)
lexicon_compiler.load_l_align_from_file(align_lexicon_fst_path)
lexicon_compiler.word_table = pywrapfst.SymbolTable.read_text(words_symbol_path)
lexicon_compiler.phone_table = pywrapfst.SymbolTable.read_text(phone_symbol_path)

In [3]:
working_directory = r"D:\temp\MFA\english\triphone"

tree_path = os.path.join(working_directory, "tree")
model_path = os.path.join(working_directory, "final.mdl")

compiler = TrainingGraphCompiler(
    model_path,
    tree_path,
    lexicon_compiler,
)


In [5]:

transcript_batch = [[compiler.to_int(x) for x in "sharma".split()]]
accep = pynini.accep("sharma", token_type=lexicon_compiler.word_table)
word_fst = pynini_to_kaldi(accep)
print(transcript_batch)

[[60656]]


In [6]:

fsts = compiler.compiler.CompileGraphsFromText(transcript_batch)
print(compiler.lexicon_compiler.disambiguation_symbols)

[103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280]


In [11]:

new_fst = VectorFst()
TableCompose(lexicon_compiler.kaldi_fst, word_fst, new_fst)

In [13]:
pynew = kaldi_to_pynini(new_fst)
pynew.set_input_symbols(lexicon_compiler.phone_table)
pynew.set_output_symbols(lexicon_compiler.word_table)
print(pynew)

0	1	<eps>	<eps>	0.127833
0	2	sil	<eps>	2.12026
1	3	ʃ	sharma	0.0100503
1	4	ʃ	sharma	0.0100503
2	3	ʃ	sharma	0.0100503
2	4	ʃ	sharma	0.0100503
3	5	ɑ	<eps>
4	6	ɑː	<eps>
5	7	ɹ	<eps>
6	8	m	<eps>
7	9	m	<eps>
8	10	ə	<eps>
9	11	ə	<eps>
10	12	<eps>	<eps>	0.248461
10	13	sil	<eps>	1.51413
11	12	<eps>	<eps>	0.174353
11	13	sil	<eps>	1.83258
12	1.91959
13	0.0100503



In [4]:
transcript = "he arrives at that spot where the break in the timber brings the house in view here he makes a halt"
transcript = [compiler.to_int(x) for x in transcript.split()]
words = " ".join([compiler.word_table.find(x) for x in transcript])
word_fst = pynini.accep(
    words,
    token_type=compiler.word_table
)

In [6]:
print(t)

1342


In [9]:

lg = pynini.compose(compiler.lexicon_compiler.fst, word_fst)

states_to_delete = set()
for i in range(lg.num_states()):
    print("STATE", i, i == lg.start())
    arc_iter = lg.arcs(i)
    while not arc_iter.done():
        arc = arc_iter.value()
        print(arc.ilabel, arc.olabel,arc.nextstate, repr(lg.final(arc.nextstate)))
        if i == lg.start() and arc.ilabel == 0:
            states_to_delete.add(arc.nextstate)
            print(states_to_delete)
        weight = lg.final(arc.nextstate)
        print(pywrapfst.Weight.zero(lg.weight_type()))
        if weight != pywrapfst.Weight.zero(lg.weight_type()) and arc.ilabel == 0:
            print(states_to_delete)
            states_to_delete.add(arc.nextstate)
        arc_iter.next()
states_to_delete = sorted(states_to_delete)
print(states_to_delete)
lg = lg.delete_states(states_to_delete)

STATE 0 True
0 0 1 <tropical Weight Infinity at 0x1d66bbaca90>
{1}
Infinity
1 0 2 <tropical Weight Infinity at 0x1d66bbaca10>
Infinity
STATE 1 False
23 31605 3 <tropical Weight Infinity at 0x1d66bbaca90>
Infinity
58 31605 4 <tropical Weight Infinity at 0x1d66bbaca10>
Infinity
58 31605 5 <tropical Weight Infinity at 0x1d66bbac8b0>
Infinity
STATE 2 False
23 31605 3 <tropical Weight Infinity at 0x1d66bbaca50>
Infinity
58 31605 4 <tropical Weight Infinity at 0x1d66bbac990>
Infinity
58 31605 5 <tropical Weight Infinity at 0x1d66bbaca90>
Infinity
STATE 3 False
0 0 6 <tropical Weight Infinity at 0x1d66bbaca50>
Infinity
1 0 7 <tropical Weight Infinity at 0x1d66bbaca90>
Infinity
STATE 4 False
24 0 8 <tropical Weight Infinity at 0x1d66bbaca10>
Infinity
STATE 5 False
23 0 9 <tropical Weight Infinity at 0x1d66bbac8b0>
Infinity
STATE 6 False
69 4063 10 <tropical Weight Infinity at 0x1d66bbaca10>
Infinity
87 4063 11 <tropical Weight Infinity at 0x1d66bbac8b0>
Infinity
STATE 7 False
69 4063 10 <tropi

In [26]:
lg.set_input_symbols(lexicon_compiler.phone_table)
lg.set_output_symbols(lexicon_compiler.word_table)
print(lg)

0	1	sil	<eps>	0.693147
1	2	i	he
1	3	iː	he
1	4	ç	he
1	5	ç	he
2	6	<eps>	<eps>	0.693147
2	7	sil	<eps>	0.693147
3	6	<eps>	<eps>	0.693147
3	7	sil	<eps>	0.693147
4	8	i	<eps>
5	9	iː	<eps>
6	10	ə	arrives
6	11	ɹ	arrives
7	10	ə	arrives
7	11	ɹ	arrives
8	6	<eps>	<eps>	0.693147
8	7	sil	<eps>	0.693147
9	6	<eps>	<eps>	0.693147
9	7	sil	<eps>	0.693147
10	12	ɹ	<eps>
11	13	aj	<eps>
12	14	aj	<eps>
13	15	v	<eps>
14	16	v	<eps>
15	17	z	<eps>
16	18	z	<eps>
17	19	<eps>	<eps>	0.693147
17	20	sil	<eps>	0.693147
18	19	<eps>	<eps>	0.693147
18	20	sil	<eps>	0.693147
19	21	a	at
19	22	a	at
19	23	a	at
20	21	a	at
20	22	a	at
20	23	a	at
21	24	t	<eps>
22	25	<eps>	<eps>	0.693147
22	26	sil	<eps>	0.693147
23	27	ʔ	<eps>
24	25	<eps>	<eps>	0.693147
24	26	sil	<eps>	0.693147
25	28	d̪	that
25	29	d̪	that
25	30	d̪	that
25	31	ð	that
25	32	ð	that
25	33	ð	that
26	28	d̪	that
26	29	d̪	that
26	30	d̪	that
26	31	ð	that
26	32	ð	that
26	33	ð	that
27	25	<eps>	<eps>	0.693147
27	26	sil	<eps>	0.693147
28	34	a	<eps>
29	35	a	<eps>
30	36	a	<eps>
31	37

True

In [7]:

lg = pynini.compose(compiler.lexicon_compiler.fst, word_fst)
t = pynini_to_kaldi(lg)
out_fst = compiler.compiler.CompileGraphFromLG(t)
print(t.Start())
print(t.NumStates())
print(out_fst.NumStates())

0
174
0


In [8]:
word_fst = VectorFst()
MakeLinearAcceptor(transcript, word_fst)
print(word_fst.NumStates())
out_fst = VectorFst()
compiler.compiler.CompileGraph(word_fst, out_fst)
print(out_fst.NumStates())

22
1458


In [6]:
word_fst = VectorFst()
phone2word_fst = VectorFst()
MakeLinearAcceptor(transcript, word_fst)
print(word_fst.NumStates())
TableCompose(lexicon_compiler.kaldi_fst, word_fst, phone2word_fst)
print(phone2word_fst.NumStates())
#out_fst = VectorFst()
out_fst = compiler.compiler.CompileGraphFromLG(phone2word_fst)
print(out_fst.NumStates())


22
174
0


In [14]:
fst = fsts[0]
print(fst)
print(fst.NumStates())

212


In [28]:
pyfst = kaldi_to_pynini(fst)
print(pyfst)

0	77	2	0	-1.02441
0	78	3	0	-1.02441
0	79	4	0	-1.02441
0	4	16074	60656	-0.566406
0	80	16075	60656	-0.566406
0	81	16076	60656	-0.566406
1	82	6	0
1	83	7	0
1	84	8	0
2	85	9	0
2	86	11	0
2	87	12	0
3	88	13	0
3	89	14	0
3	90	16	0
4	91	16098	0
5	92	16132	0
6	93	11140	0
7	94	11179	0
8	42	15806	0
8	95	15807	0
8	96	15808	0
9	97	11345	0
10	98	11358	0
11	12	4752	0
11	99	4753	0
11	100	4754	0
12	101	4850	0
13	102	4909	0
14	15	12632	0	-0.117188
14	16	12633	0	-0.117188
14	17	12634	0	-0.117188
15	40	13028	0	0.117188
15	41	13062	0	2.20215
16	38	13132	0	0.117188
16	39	13176	0	2.20215
17	103	2	0	-0.163086
17	104	3	0	-0.163086
17	105	4	0	-0.163086
17	2.20215
18	106	6	0
18	107	7	0
18	108	8	0
19	109	9	0
19	110	11	0
19	111	12	0
20	112	13	0
20	113	14	0
20	114	16	0
21	22	18	0
22	115	2	0	0.0859375
22	116	3	0	0.0859375
22	117	4	0	0.0859375
22	22	17	0
22	1.40918
23	118	6	0
23	119	7	0
23	120	8	0
24	121	9	0
24	122	11	0
24	123	12	0
25	124	13	0
25	125	14	0
25	126	16	0
26	27	18	0
27	127	2	0	-0.00976562
27	128	3	0	-0.009765

In [19]:
pyfst.set_input_symbols(lexicon_compiler.phone_table)
pyfst.set_output_symbols(lexicon_compiler.word_table)
print(pyfst)

0	77	spn	<eps>	-1.02441
0	78	a	<eps>	-1.02441
0	79	aj	<eps>	-1.02441
0	4	?	sharma	-0.566406
0	80	?	sharma	-0.566406
0	81	?	sharma	-0.566406
1	82	aː	<eps>
1	83	b	<eps>
1	84	bʲ	<eps>
2	85	c	<eps>
2	86	cʷ	<eps>
2	87	d	<eps>
3	88	dʒ	<eps>
3	89	dʲ	<eps>
3	90	e	<eps>
4	91	?	<eps>
5	92	?	<eps>
6	93	?	<eps>
7	94	?	<eps>
8	42	?	<eps>
8	95	?	<eps>
8	96	?	<eps>
9	97	?	<eps>
10	98	?	<eps>
11	12	?	<eps>
11	99	?	<eps>
11	100	?	<eps>
12	101	?	<eps>
13	102	?	<eps>
14	15	?	<eps>	-0.117188
14	16	?	<eps>	-0.117188
14	17	?	<eps>	-0.117188
15	40	?	<eps>	0.117188
15	41	?	<eps>	2.20215
16	38	?	<eps>	0.117188
16	39	?	<eps>	2.20215
17	103	spn	<eps>	-0.163086
17	104	a	<eps>	-0.163086
17	105	aj	<eps>	-0.163086
17	2.20215
18	106	aː	<eps>
18	107	b	<eps>
18	108	bʲ	<eps>
19	109	c	<eps>
19	110	cʷ	<eps>
19	111	d	<eps>
20	112	dʒ	<eps>
20	113	dʲ	<eps>
20	114	e	<eps>
21	22	eː	<eps>
22	115	spn	<eps>	0.0859375
22	116	a	<eps>	0.0859375
22	117	aj	<eps>	0.0859375
22	22	ej	<eps>
22	1.40918
23	118	aː	<eps>
23	119	b	<eps>
23	120

In [4]:
accep = pynini.accep("sharma", token_type=lexicon_compiler.word_table)
l_accep = pynini.compose(lexicon_compiler.fst, accep)

l_accep.set_input_symbols(lexicon_compiler.phone_table)
l_accep.set_output_symbols(lexicon_compiler.word_table)
print(l_accep)

0	1	<eps>	<eps>	0.693147
0	2	sil	<eps>	0.693147
1	3	ʃ	sharma
1	4	ʃ	sharma
1	5	ʃ	sharma
2	3	ʃ	sharma
2	4	ʃ	sharma
2	5	ʃ	sharma
3	6	ɑ	<eps>
4	7	aː	<eps>
5	8	ɑː	<eps>
6	9	ɹ	<eps>
7	10	m	<eps>
8	11	m	<eps>
9	12	m	<eps>
10	13	ə	<eps>
11	14	ə	<eps>
12	15	ə	<eps>
13	16	<eps>	<eps>	0.693147
13	17	sil	<eps>	0.693147
14	16	<eps>	<eps>	0.693147
14	17	sil	<eps>	0.693147
15	16	<eps>	<eps>	0.693147
15	17	sil	<eps>	0.693147
16
17



In [33]:
feature_path = r"D:\temp\MFA\english\english\split10\final_features.6.ark"
lda_path = r"D:\temp\MFA\english\sat_3_ali\lda.mat"
feature_archive = FeatureArchive(feature_path, lda_mat_file_name=lda_path)
feats = feature_archive["23985-21732"]

In [34]:
print(feats.NumCols(), feats.NumRows())

40 194


In [35]:

aligner = GmmAligner(
    model_path,
    silence_phones=[1],
)

In [38]:
alignment = aligner.align_utterance(fst, feats)
print(alignment)

Alignment(utterance_id=None, alignment=[2, 8, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 18, 17, 17, 17, 17, 17, 17, 17, 17, 17, 4, 14, 15, 15, 15, 15, 12, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 18, 16059, 16104, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16103, 16132, 11228, 11345, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11344, 11358, 4752, 4850, 4849, 4849, 4909, 12632, 13028, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13027, 13132, 3, 12, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 1

In [40]:
alignment.generate_ctm(aligner.transition_model, lexicon_compiler.phone_table)

[<CtmInterval of labeled 'sil(1)' from 0.000000 to 0.490000 confidence=-23.331520>,
 <CtmInterval of labeled 'sil(1)' from 0.490000 to 0.680000 confidence=-41.613514>,
 <CtmInterval of labeled 'ʃ(91)' from 0.680000 to 0.900000 confidence=-51.545914>,
 <CtmInterval of labeled 'ɑː(63)' from 0.900000 to 1.130000 confidence=-37.119740>,
 <CtmInterval of labeled 'm(31)' from 1.130000 to 1.180000 confidence=-49.342308>,
 <CtmInterval of labeled 'ə(69)' from 1.180000 to 1.510000 confidence=-45.984776>,
 <CtmInterval of labeled 'sil(1)' from 1.510000 to 1.940000 confidence=-40.352222>]

In [41]:
alignment = aligner.align_utterance(fst, feats, boost_silence=1.5)

In [42]:
alignment.generate_ctm(aligner.transition_model, lexicon_compiler.phone_table)

[<CtmInterval of labeled 'sil(1)' from 0.000000 to 0.680000 confidence=-28.164497>,
 <CtmInterval of labeled 'ʃ(91)' from 0.680000 to 0.900000 confidence=-51.545914>,
 <CtmInterval of labeled 'ɑː(63)' from 0.900000 to 1.130000 confidence=-37.119740>,
 <CtmInterval of labeled 'm(31)' from 1.130000 to 1.180000 confidence=-49.342308>,
 <CtmInterval of labeled 'ə(69)' from 1.180000 to 1.450000 confidence=-46.747826>,
 <CtmInterval of labeled 'sil(1)' from 1.450000 to 1.940000 confidence=-40.260025>]

In [51]:
alignment = aligner.align_utterance(fst, feats, boost_silence=1.25)
alignment.generate_ctm(aligner.transition_model, lexicon_compiler.phone_table)

[<CtmInterval of labeled 'sil(1)' from 0.000000 to 0.680000 confidence=-28.675505>,
 <CtmInterval of labeled 'ʃ(91)' from 0.680000 to 0.900000 confidence=-51.545914>,
 <CtmInterval of labeled 'ɑː(63)' from 0.900000 to 1.130000 confidence=-37.119740>,
 <CtmInterval of labeled 'm(31)' from 1.130000 to 1.180000 confidence=-49.342308>,
 <CtmInterval of labeled 'ə(69)' from 1.180000 to 1.510000 confidence=-45.984776>,
 <CtmInterval of labeled 'sil(1)' from 1.510000 to 1.940000 confidence=-40.143929>]

In [ ]:
alignment.generate_word_ctm()